# Notebook 02 — Preprocessing & Train / Val / Test Split


Right now all 780 images are sitting in 3 folders:
```
data/raw/benign/       437 images
data/raw/malignant/    210 images
data/raw/normal/       133 images
```

We need to split them into 3 separate groups:
```
data/processed/train/   70%  ->  546 images  ->  model LEARNS from this
data/processed/val/     15%  ->  117 images  ->  we TUNE and MONITOR with this
data/processed/test/    15%  ->  117 images  ->  final EVALUATION only
```

## Why do we need 3 splits?

- TRAIN  → The model sees these images and adjusts its weights
- VAL    → We check after every epoch if the model is improving or overfitting.
           We also use this to pick the best hyperparameters (learning rate etc.)
- TEST   → We touch this ONLY ONCE at the very end to get the final result.
           This gives us an honest score that was never used in any decision.

## Important rule
NEVER look at test results, adjust something, then re-test.
That would be cheating — the test set must stay completely unseen.

---
##  Import Libraries

In [ ]:
import os
import shutil    # shutil.copy() copies a file from one folder to another
import random    # random.shuffle() shuffles a list randomly
from pathlib import Path

print('Libraries imported successfully!')

---
## Configure Settings

All the settings are in one place so they are easy to change.

SEED = 42 means the shuffle is random but REPRODUCIBLE.
If you run this script again tomorrow, you get exactly the same split.
Write this seed number in your report Methods section.

In [ ]:
# Where the original images are
SRC_DIR  = Path('data/raw')

# Where the split images will be saved
DEST_DIR = Path('data/processed')

# The three class folders
CLASSES  = ['benign', 'malignant', 'normal']

# Split proportions — must add up to 1.0
TRAIN_RATIO = 0.70   # 70 percent
VAL_RATIO   = 0.15   # 15 percent
TEST_RATIO  = 0.15   # 15 percent

# Fixed random seed for reproducibility
SEED = 42

print(f'Source folder     :  {SRC_DIR}')
print(f'Destination folder:  {DEST_DIR}')
print(f'Split ratio       :  {TRAIN_RATIO} / {VAL_RATIO} / {TEST_RATIO}')
print(f'Random seed       :  {SEED}')

---
##  Verify Source Folders Exist

Before doing anything, we check that all 3 class folders
are present and have the right number of images.

In [ ]:
print('Checking source folders...')
print('-' * 40)

all_ok = True

for cls in CLASSES:
    folder = SRC_DIR / cls
    
    if not folder.exists():
        print(f'NOT FOUND  ->  {folder}')
        all_ok = False
    else:
        count = len(list(folder.glob('*.png')))
        print(f'OK  ->  {folder}  ({count} images)')

print('-' * 40)

if all_ok:
    print('All folders found. Ready to split!')
else:
    print('ERROR: Fix the missing folders before continuing.')

---
##  Create the Split

This is the main step. Here is what happens for EACH class:

Step 1 — List all .png images in the class folder
Step 2 — Shuffle them randomly (using fixed seed)
Step 3 — Slice the list into train / val / test groups
Step 4 — Copy each image to its new folder

We do this PER CLASS separately — this is called a STRATIFIED split.
It guarantees that each split has the same proportion of each class.

Example for benign (437 images):
```
  n_train = int(437 x 0.70) = 305
  n_val   = int(437 x 0.15) = 65
  n_test  = 437 - 305 - 65  = 67   (remainder so no images are lost)
```

In [ ]:
# Fix the random seed BEFORE any shuffling
random.seed(SEED)

# We will store the counts here to print a summary later
# Example structure: counts['train']['benign'] = 305
counts = {
    'train': {},
    'val':   {},
    'test':  {}
}

# ── Loop through each class ───────────────────────────────────
for cls in CLASSES:
    print(f'Processing class: {cls}')

    # STEP 1: Get all image paths in this class folder
    all_images = sorted(list((SRC_DIR / cls).glob('*.png')))
    n_total    = len(all_images)

    # STEP 2: Shuffle the list randomly
    # After shuffling, the order is random but the same every time
    # because we set random.seed(SEED) above
    random.shuffle(all_images)

    # STEP 3: Calculate how many images go in each split
    n_train = int(n_total * TRAIN_RATIO)
    n_val   = int(n_total * VAL_RATIO)
    n_test  = n_total - n_train - n_val   # remainder goes to test

    # Slice the shuffled list into 3 groups
    train_images = all_images[ : n_train]                    # first 70%
    val_images   = all_images[n_train : n_train + n_val]     # next 15%
    test_images  = all_images[n_train + n_val : ]            # last 15%

    # STEP 4: Copy each image to its new destination folder
    for split_name, image_list in [('train', train_images),
                                    ('val',   val_images),
                                    ('test',  test_images)]:

        # Create the destination folder if it does not exist yet
        # Example: data/processed/train/benign/
        dest_folder = DEST_DIR / split_name / cls
        dest_folder.mkdir(parents=True, exist_ok=True)

        # Copy every image in this group to the destination folder
        for img_path in image_list:
            shutil.copy(img_path, dest_folder / img_path.name)

        # Save the count for the summary
        counts[split_name][cls] = len(image_list)

    print(f'  train={n_train}   val={n_val}   test={n_test}   total={n_total}')

print()
print('Split complete!')

---
## Print Summary Table

We print a table to verify everything looks correct.
Use these numbers in your report Methods section.

In [ ]:
print('=' * 58)
print(f'  {"Split":<8}  {"Benign":>8}  {"Malignant":>10}  {"Normal":>8}  {"Total":>7}')
print('=' * 58)

grand_total = 0

for split_name in ['train', 'val', 'test']:
    b = counts[split_name]['benign']
    m = counts[split_name]['malignant']
    n = counts[split_name]['normal']
    t = b + m + n
    grand_total += t
    print(f'  {split_name:<8}  {b:>8}  {m:>10}  {n:>8}  {t:>7}')

print('=' * 58)
print(f'  {"TOTAL":<8}  '
      f'{sum(counts[s]["benign"]    for s in ["train","val","test"]):>8}  '
      f'{sum(counts[s]["malignant"] for s in ["train","val","test"]):>10}  '
      f'{sum(counts[s]["normal"]    for s in ["train","val","test"]):>8}  '
      f'{grand_total:>7}')
print('=' * 58)
print()
print('Expected:')
print('  train  ->  ~306 benign   ~147 malignant   ~93 normal   = ~546 total')
print('  val    ->   ~65 benign    ~32 malignant   ~20 normal   = ~117 total')
print('  test   ->   ~66 benign    ~31 malignant   ~20 normal   = ~117 total')

---
## Verify the Folder Structure Was Created Correctly

We physically count the files in each output folder
to confirm the copy worked correctly.

In [ ]:
print('Verifying output folders...')
print()

for split_name in ['train', 'val', 'test']:
    print(f'  {split_name}/')
    for cls in CLASSES:
        folder = DEST_DIR / split_name / cls
        count  = len(list(folder.glob('*.png')))
        print(f'    {cls:<12}  ->  {count} images')
    print()

print('Folder structure:')
print()
print('  data/processed/')
print('    train/')
print('      benign/')
print('      malignant/')
print('      normal/')
print('    val/')
print('      benign/')
print('      malignant/')
print('      normal/')
print('    test/')
print('      benign/')
print('      malignant/')
print('      normal/')

---
##  Test That PyTorch Can Read the Split

ImageFolder is a PyTorch tool that automatically:
- Reads all images from a folder
- Assigns a label based on the subfolder name
  (benign=0, malignant=1, normal=2)

We test all 3 splits to confirm everything is working.

In [ ]:
from torchvision import datasets, transforms

# A minimal transform just for testing
# We only resize and convert to tensor — no augmentation yet
basic_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # resize all images to 224x224
    transforms.ToTensor()            # convert image to PyTorch tensor
])

# Load each split using ImageFolder
# ImageFolder reads the subfolders as class labels automatically
train_dataset = datasets.ImageFolder('data/processed/train', transform=basic_transform)
val_dataset   = datasets.ImageFolder('data/processed/val',   transform=basic_transform)
test_dataset  = datasets.ImageFolder('data/processed/test',  transform=basic_transform)

print('Dataset loaded successfully!')
print()
print(f'Class names detected : {train_dataset.classes}')
print(f'Class to index map   : {train_dataset.class_to_idx}')
print()
print(f'Train dataset size   : {len(train_dataset)} images')
print(f'Val   dataset size   : {len(val_dataset)}   images')
print(f'Test  dataset size   : {len(test_dataset)}   images')

# Check one sample to make sure shapes are correct
sample_image, sample_label = train_dataset[0]
print()
print(f'Sample image shape   : {sample_image.shape}')
print(f'  Expected           : torch.Size([3, 224, 224])')
print(f'  3 = channels (RGB), 224 = height, 224 = width')
print(f'Sample label         : {sample_label}  ({train_dataset.classes[sample_label]})')

---
##  Final Summary

Everything is ready. Use this information in your report.

In [ ]:
print('=' * 55)
print('  PREPROCESSING SUMMARY')
print('=' * 55)
print(f'  Split ratio        :  70% train / 15% val / 15% test')
print(f'  Random seed        :  {SEED}  (reproducible split)')
print(f'  Split method       :  Stratified per class')
print(f'  Train images       :  {len(train_dataset)}')
print(f'  Val   images       :  {len(val_dataset)}')
print(f'  Test  images       :  {len(test_dataset)}')
print(f'  Total images       :  {len(train_dataset)+len(val_dataset)+len(test_dataset)}')
print(f'  Image resize       :  224 x 224 pixels')
print(f'  Output format      :  PyTorch ImageFolder compatible')
print('=' * 55)
print()
print('NEXT STEP:  Run  03_Model_Training.ipynb')
print('That notebook loads these splits and trains the ResNet-18 model.')